# 类继承关系
通过访问器机制为 Pandas 对象提供 Plotly Express 绘图功能。
```mermaid
classDiagram
    class BaseAccessor
    class PXAccessor
    class BaseSRAccessor
    class PXSRAccessor
    class BaseDFAccessor
    class PXDFAccessor

    
    %% 继承关系
    BaseAccessor <|-- PXAccessor

    BaseSRAccessor <|-- PXSRAccessor
    PXAccessor <|-- PXSRAccessor

    PXAccessor <|-- PXDFAccessor
    BaseDFAccessor <|-- PXDFAccessor
```

# attach_px_methods
用于装饰类，例如
```python
@attach_px_methods
class A:
    pass
```
`A` 被装饰后拥有了 *Plotly Express* 的绘图方法。

## 源码

```python
def attach_px_methods(cls: tp.Type[tp.T]) -> tp.Type[tp.T]:

    for px_func_name, px_func in getmembers(px, isfunction):
        if checks.func_accepts_arg(px_func, 'data_frame') or px_func_name == 'imshow':
            def plot_func(self, *args, _px_func_name: str = px_func_name,
                          _px_func: tp.Callable = px_func, **kwargs) -> tp.BaseFigure:
                from vectorbt._settings import settings
                layout_cfg = settings['plotting']['layout']

                layout_kwargs = dict(
                    template=kwargs.pop('template', layout_cfg['template']),
                    width=kwargs.pop('width', layout_cfg['width']),
                    height=kwargs.pop('height', layout_cfg['height'])
                )
                # Fix category_orders
                if 'color' in kwargs:
                    if isinstance(kwargs['color'], str):
                        if isinstance(self.obj, pd.DataFrame):
                            if kwargs['color'] in self.obj.columns:
                                category_orders = dict()
                                category_orders[kwargs['color']] = sorted(self.obj[kwargs['color']].unique())
                                kwargs = merge_dicts(dict(category_orders=category_orders), kwargs)

                # Fix Series name
                obj = self.obj.copy(deep=False)
                if isinstance(obj, pd.Series):
                    if obj.name is not None:
                        obj = obj.rename(str(obj.name))
                else:
                    obj.columns = clean_labels(obj.columns)
                obj.index = clean_labels(obj.index)

                if _px_func_name == 'imshow':
                    return make_figure(_px_func(
                        to_2d_array(obj), *args, **layout_kwargs, **kwargs
                    ), layout=layout_kwargs)
                return make_figure(_px_func(
                    obj, *args, **layout_kwargs, **kwargs
                ), layout=layout_kwargs)

            setattr(cls, px_func_name, plot_func)
    return cls
```

## 例子

In [ ]:
import pandas as pd
from vectorbt.px_accessors import attach_px_methods, BaseAccessor

data = pd.DataFrame({
    'col1': [1, 2, 3, 4, 5],
    'col2': [10, 20, 30, 40, 50]
})

@attach_px_methods
class MyPlotAccessor(BaseAccessor):
    pass

# 现在MyPlotAccessor自动拥有所有px绘图方法
accessor = MyPlotAccessor(data)
fig = accessor.bar()  # 调用px.bar()方法
fig = accessor.scatter(x='col1', y='col2')  # 调用px.scatter()方法
fig.show()

# class PXAccessor(BaseAccessor)
类 `PXAccessor` 相当于添加了 *Plotly Express* 绘图功能的 `BaseAccessor`。

```python
@attach_px_methods
class PXAccessor(BaseAccessor):

    def __init__(self, obj: tp.SeriesFrame, **kwargs) -> None:
        BaseAccessor.__init__(self, obj, **kwargs)
```

# class PXSRAccessor(PXAccessor, BaseSRAccessor)
对于 `Vbt_SRAccessor` 类型的实例 `obj`，`obj.px` 相当于
- `PXSRAccessor(obj)`

另外参考 [root_accessors.ipynb](./root_accessors.ipynb)，对于任意 `Series` 的实例 `obj`
- `obj.vbt` 相当于 `Vbt_SRAccessor(obj)`
- 于是 `obj.vbt.px` 相当于 `PXSRAccessor(Vbt_SRAccessor(obj))`

```python
@register_series_vbt_accessor('px')
class PXSRAccessor(PXAccessor, BaseSRAccessor):

    def __init__(self, obj: tp.Series, **kwargs) -> None:
        BaseSRAccessor.__init__(self, obj, **kwargs)
        PXAccessor.__init__(self, obj, **kwargs)
```

# class PXDFAccessor(PXAccessor, BaseDFAccessor)
对于 `Vbt_DFAccessor` 类型的实例 `obj`，`obj.px` 相当于
- `PXDFAccessor(obj)`

另外参考 [root_accessors.ipynb](./root_accessors.ipynb)，对于任意 `DataFrame` 的实例 `obj`
- `obj.vbt` 相当于 `Vbt_DFAccessor(obj)`
- 于是 `obj.vbt.px` 相当于 `PXDFAccessor(Vbt_DFAccessor(obj))`

```python
@register_dataframe_vbt_accessor('px')
class PXDFAccessor(PXAccessor, BaseDFAccessor):

    def __init__(self, obj: tp.Frame, **kwargs) -> None:
        BaseDFAccessor.__init__(self, obj, **kwargs)
        PXAccessor.__init__(self, obj, **kwargs)
```